In [ ]:
# Cài & chạy ollama nền
!curl -fsSL https://ollama.com/install.sh | sh
# chạy server nền, mở cổng cho mọi nguồn (CORS)
import os
%env OLLAMA_HOST=0.0.0.0
%env OLLAMA_ORIGINS=*
!nohup ollama serve >/dev/null 2>&1 &

In [ ]:
# Check server đã lên
import time, urllib.request
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=1)
        print("Ollama is up")
        break
    except Exception:
        time.sleep(1)

In [ ]:
!ollama pull gpt-oss:20b
!ollama list
!curl -s http://127.0.0.1:11434/api/tags
# test nhanh
!curl -s http://127.0.0.1:11434/api/generate -d '{"model":"gpt-oss:20b","prompt":"xin chào","stream":false}'

In [ ]:
!ollama pull qwen2.5:7b-instruct-q4_0
!curl http://localhost:11434/api/generate -d '{"model": "qwen2.5:7b-instruct-q4_0","prompt": "Who are you?","stream": false}'

In [ ]:
# Start ngrok & get the public URL
!pip -q install pyngrok
from google.colab import userdata
from pyngrok import ngrok

ngrok_api_key = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(ngrok_api_key)

# Mặc định Ollama chạy ở port này
tunnel = ngrok.connect(11434, "http")
print("Public URL:", tunnel.public_url)

In [ ]:
!curl https://taunya-hexahydrated-hezekiah.ngrok-free.dev/api/generate -d '{"model":"gpt-oss:20b","prompt":"hi","stream":false}'

In [ ]:
!pip install fastapi nest-asyncio pyngrok uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

@app.get('/')
async def root():
    return {'hello': 'world'}

@app.get('/ask')
async def ask(question: str):
    return {
        'question': question,
        'answer': "I don't know"
    }

import threading
import uvicorn

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("Server started on http://0.0.0.0:8000")

In [ ]:
!curl localhost:8000

In [ ]:
!curl localhost:8000/ask?question=Tell%20me%20a%20joke